# Weather Agent Debugging, Fixes & Multi-Scenario Verification

## 1. Project Architecture & Directory Structure

Based on the project layout (`final_assignment/`), the Weather Agent is a modular LangGraph application structured as follows:

```text
final_assignment/
├── components/
│   ├── __init__.py
│   ├── config.py             # API endpoints, timeouts, temperature thresholds, WMO codes
│   ├── helper_functions.py   # Temperature classification, WMO lookup, greetings, UTC/local time formatting
│   ├── nodes.py              # LangGraph node functions (fetch_location_data, fetch_weather_data, generate_weather_info)
│   ├── schema.py             # Pydantic models for LocationData and WeatherData validation
│   └── state.py              # WeatherAgentState TypedDict state definition
├── graph.py                  # StateGraph construction, edge wiring, and compilation
├── main.py                   # CLI entry point invoking the compiled weather_agent
├── requirements.txt          # Python package dependencies
└── weather_agent_debugging.ipynb
```

---

## 2. Comprehensive Bug Report & Systematic Fixes

| # | File | Location | Bug Description | Impact / Symptom | Fix Applied |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | `components/config.py` | Line 53 | `WEATHER_API_BASE_URL: str = ""` redeclared at the bottom of `Config`, overwriting the valid URL (`"https://api.open-meteo.com/v1/forecast"`) on Line 14. | `requests.get()` fails with `MissingSchema: Invalid URL ''` when fetching weather data. | Removed the duplicate `WEATHER_API_BASE_URL: str = ""` attribute. |
| **2** | `components/config.py` | Line 55 | `TEMP_MIN = str = "Needs Debugging"` uses invalid chained assignment instead of a type annotation and introduces a placeholder string. | Causes Pydantic field definition issues and feeds `"Needs Debugging"` into `classify_temperature()`. | Removed `TEMP_MIN = str = "Needs Debugging"`. |
| **3** | `components/helper_functions.py` | Lines 14–15 | `if temp_celsius: return config.TEMP_MIN` placed at the top of `classify_temperature()`. | Any non-zero temperature evaluates as truthy (`True`) and returns `"Needs Debugging"` instead of `"cold"`, `"cool"`, `"comfortable"`, `"warm"`, or `"hot"`. | Removed the `if temp_celsius:` branch and started directly with `if temp_celsius < config.TEMP_COLD: return "cold"`. |
| **4** | `components/nodes.py` | Line 33 & Line 119 | `fetch_location_data()` checks `'country'` in `required_fields`, whereas `generate_weather_info()` reads `location["country_name"]` (matching `schema.py`). | Inconsistent key validation if a payload includes `'country'` without `'country_name'`. | Validated `'country_name'` in `required_fields` and used `location.get("country_name") or location.get("country")` in `generate_weather_info()`. |
| **5** | `components/nodes.py` | Line 37 | `state["location_data"] = {}` assigns an empty dict instead of the parsed `location_data` JSON response. | Downstream nodes see an empty (falsy) `state["location_data"]` and raise `Exception("Location data not available for weather fetch")`. | Assigned `state["location_data"] = location_data`. |
| **6** | `components/nodes.py` | Lines 124 & 144 | `# wind_unit = units.get("windspeed", "km/h")` is commented out, and the f-string has a hardcoded literal `"• Wind: {windspeed} wind_unit"`. | Output displays literal text `wind_unit` (e.g., `• Wind: 12.5 wind_unit`) instead of the actual unit (`km/h`). | Uncommented `wind_unit = units.get("windspeed", "km/h")` and updated the f-string to `f"• Wind: {windspeed} {wind_unit}"`. |
| **7** | `graph.py` | Lines 18–20 | `builder.add_edge("fetch_location_data", "generate_weather_info")` bypasses the `fetch_weather_data` node completely. | `fetch_weather_data` never runs; `state["weather_data"]` stays `None`, causing `generate_weather_info` to crash. | Wired sequential edges: `START -> fetch_location_data -> fetch_weather_data -> generate_weather_info -> END`. |
| **8** | `graph.py` | Line 23 | `weather_agent = builder` exports the uncompiled `StateGraph` builder object instead of compiling it. | `weather_agent.invoke(state)` in `main.py` raises `AttributeError: 'StateGraph' object has no attribute 'invoke'`. | Changed to `weather_agent = builder.compile()`. |
| **9** | `main.py` | Line 31 | Entry guard written with single underscores: `if __name__ == "_main_":`. | Running `python main.py` exits silently without calling `main()` because `__name__` is `"__main__"`. | Fixed entry guard to `if __name__ == "__main__":`. |
| **10** | `components/schema.py` & `requirements.txt` | `CurrentWeather.is_day` & dependencies | `CurrentWeather.is_day` typed as `bool` while Open-Meteo returns `int` (`0`/`1`), and `requirements.txt` omitted `requests` and `python-dotenv`. | Potential strict type mismatch and missing runtime dependencies in clean virtual environments. | Updated `is_day: int` in `schema.py` and added `requests` and `python-dotenv` to `requirements.txt`. |

## 3. Side-by-Side Code Fixes

### 3.1 `components/config.py`
```python
# ❌ BEFORE (Buggy)
    WEATHER_API_BASE_URL: str = ""
    TEMP_MIN = str = "Needs Debugging"

# ✅ AFTER (Fixed)
    # Removed duplicate WEATHER_API_BASE_URL override and invalid TEMP_MIN attribute
    # Preserves WEATHER_API_BASE_URL: str = "https://api.open-meteo.com/v1/forecast"
```

### 3.2 `components/helper_functions.py`
```python
# ❌ BEFORE (Buggy)
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius:
        return config.TEMP_MIN
    elif temp_celsius < config.TEMP_COLD:
        return "cold"
    ...

# ✅ AFTER (Fixed)
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius < config.TEMP_COLD:
        return "cold"
    elif temp_celsius < config.TEMP_COOL:
        return "cool"
    elif temp_celsius < config.TEMP_COMFORTABLE:
        return "comfortable"
    elif temp_celsius < config.TEMP_WARM:
        return "warm"
    else:
        return "hot"
```

### 3.3 `components/nodes.py`
```python
# ❌ BEFORE (Buggy)
        # In fetch_location_data:
        required_fields = ['city', 'region', 'country', 'latitude', 'longitude', 'utc_offset', 'timezone']
        ...
        state["location_data"] = {}

        # In generate_weather_info:
        # wind_unit = units.get("windspeed", "km/h")
        ...
        f"• Wind: {windspeed} wind_unit"

# ✅ AFTER (Fixed)
        # In fetch_location_data:
        required_fields = ['city', 'region', 'country_name', 'latitude', 'longitude', 'utc_offset', 'timezone']
        ...
        state["location_data"] = location_data

        # In generate_weather_info:
        wind_unit = units.get("windspeed", "km/h")
        ...
        f"• Wind: {windspeed} {wind_unit}"
```

### 3.4 `graph.py` & `main.py`
```python
# ❌ BEFORE (Buggy graph.py)
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)
weather_agent = builder

# ✅ AFTER (Fixed graph.py)
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "fetch_weather_data")
builder.add_edge("fetch_weather_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)
weather_agent = builder.compile()

# ❌ BEFORE (Buggy main.py)
if __name__ == "_main_":
    main()

# ✅ AFTER (Fixed main.py)
if __name__ == "__main__":
    main()
```

import os, sys
sys.path.insert(0, os.path.abspath("weather_agent"))

## 4. Unit Verification of Fixed Components

First, let's verify `components/config.py` and `components/helper_functions.py` across boundary values and formats.

In [1]:
from components.config import config
from components.helper_functions import (
    classify_temperature,
    get_weather_description,
    get_greeting,
    parse_utc_offset,
    format_local_time
)

print("1. Verified Config Endpoints:")
print(f"   LOCATION_API_URL     = {config.LOCATION_API_URL}")
print(f"   WEATHER_API_BASE_URL = {config.WEATHER_API_BASE_URL}")
assert config.WEATHER_API_BASE_URL == "https://api.open-meteo.com/v1/forecast"
assert not hasattr(config, "TEMP_MIN")

print("\n2. Verified Temperature Classifications:")
test_temps = [-5.0, 0.0, 9.9, 10.0, 15.5, 18.0, 22.4, 26.0, 30.0, 32.0, 38.5]
for t in test_temps:
    print(f"   {t:>5.1f}°C -> {classify_temperature(t)}")

print("\n3. Verified Greetings, Weather Descriptions & Time Formatting:")
print(f"   is_day=1 -> {get_greeting(1)}")
print(f"   is_day=0 -> {get_greeting(0)}")
print(f"   WMO 0    -> {get_weather_description(0)}")
print(f"   WMO 95   -> {get_weather_description(95)}")
print(f"   WMO 999  -> {get_weather_description(999)}")
print(f"   Time (+05:30) -> {format_local_time('2026-10-23T13:15', '+05:30')}")
print(f"   Time (+0530)  -> {format_local_time('2026-10-23T13:15', '+0530')}")
print(f"   Time (-08:00) -> {format_local_time('2026-10-23T18:30Z', '-08:00')}")

1. Verified Config Endpoints:
   LOCATION_API_URL     = https://ipapi.co/json/
   WEATHER_API_BASE_URL = https://api.open-meteo.com/v1/forecast

2. Verified Temperature Classifications:
    -5.0°C -> cold
     0.0°C -> cold
     9.9°C -> cold
    10.0°C -> cool
    15.5°C -> cool
    18.0°C -> comfortable
    22.4°C -> comfortable
    26.0°C -> warm
    30.0°C -> warm
    32.0°C -> hot
    38.5°C -> hot

3. Verified Greetings, Weather Descriptions & Time Formatting:
   is_day=1 -> Good morning
   is_day=0 -> Good evening
   WMO 0    -> Clear sky
   WMO 95   -> Thunderstorm
   WMO 999  -> Weather code 999
   Time (+05:30) -> 13:15 UTC | 18:45 (UTC+05:30)
   Time (+0530)  -> 13:15 UTC | 18:45 (UTC+0530)
   Time (-08:00) -> 18:30 UTC | 10:30 (UTC-08:00)

## 5. End-to-End Testing Across Multiple Scenarios

Next, we test the compiled `weather_agent` graph (`START -> fetch_location_data -> fetch_weather_data -> generate_weather_info -> END`) across five diverse real-world weather and location scenarios, plus error-handling scenarios.

In [2]:
from unittest.mock import patch, MagicMock
from graph import weather_agent
from components.state import WeatherAgentState

scenarios = [
    {
        "title": "Scenario 1: Comfortable Morning in Bengaluru, India (+0530)",
        "user_name": "Aarav",
        "location_payload": {
            "city": "Bengaluru",
            "region": "Karnataka",
            "country": "IN",
            "country_name": "India",
            "latitude": 12.9716,
            "longitude": 77.5946,
            "utc_offset": "+0530",
            "timezone": "Asia/Kolkata"
        },
        "weather_payload": {
            "latitude": 12.9716,
            "longitude": 77.5946,
            "timezone": "GMT",
            "utc_offset_seconds": 0,
            "current_weather_units": {
                "time": "iso8601",
                "temperature": "°C",
                "windspeed": "km/h",
                "winddirection": "°",
                "weathercode": "wmo code"
            },
            "current_weather": {
                "time": "2026-10-23T04:00",
                "temperature": 23.4,
                "windspeed": 11.2,
                "winddirection": 95,
                "is_day": 1,
                "weathercode": 2
            }
        }
    },
    {
        "title": "Scenario 2: Freezing Snowy Night in Reykjavik, Iceland (+00:00)",
        "user_name": "Katrin",
        "location_payload": {
            "city": "Reykjavik",
            "region": "Capital Region",
            "country": "IS",
            "country_name": "Iceland",
            "latitude": 64.1466,
            "longitude": -21.9426,
            "utc_offset": "+00:00",
            "timezone": "Atlantic/Reykjavik"
        },
        "weather_payload": {
            "latitude": 64.1466,
            "longitude": -21.9426,
            "timezone": "GMT",
            "utc_offset_seconds": 0,
            "current_weather_units": {
                "time": "iso8601",
                "temperature": "°C",
                "windspeed": "km/h",
                "winddirection": "°",
                "weathercode": "wmo code"
            },
            "current_weather": {
                "time": "2026-10-23T22:15",
                "temperature": -2.8,
                "windspeed": 28.5,
                "winddirection": 30,
                "is_day": 0,
                "weathercode": 73
            }
        }
    },
    {
        "title": "Scenario 3: Hot Daytime in Dubai, UAE (+04:00)",
        "user_name": "Zayd",
        "location_payload": {
            "city": "Dubai",
            "region": "Dubai",
            "country": "AE",
            "country_name": "United Arab Emirates",
            "latitude": 25.2048,
            "longitude": 55.2708,
            "utc_offset": "+04:00",
            "timezone": "Asia/Dubai"
        },
        "weather_payload": {
            "latitude": 25.2048,
            "longitude": 55.2708,
            "timezone": "GMT",
            "utc_offset_seconds": 0,
            "current_weather_units": {
                "time": "iso8601",
                "temperature": "°C",
                "windspeed": "km/h",
                "winddirection": "°",
                "weathercode": "wmo code"
            },
            "current_weather": {
                "time": "2026-10-23T09:30",
                "temperature": 36.2,
                "windspeed": 18.0,
                "winddirection": 310,
                "is_day": 1,
                "weathercode": 0
            }
        }
    },
    {
        "title": "Scenario 4: Cool Rainy Evening in London, UK (+01:00)",
        "user_name": "Oliver",
        "location_payload": {
            "city": "London",
            "region": "England",
            "country": "GB",
            "country_name": "United Kingdom",
            "latitude": 51.5074,
            "longitude": -0.1278,
            "utc_offset": "+01:00",
            "timezone": "Europe/London"
        },
        "weather_payload": {
            "latitude": 51.5074,
            "longitude": -0.1278,
            "timezone": "GMT",
            "utc_offset_seconds": 0,
            "current_weather_units": {
                "time": "iso8601",
                "temperature": "°C",
                "windspeed": "km/h",
                "winddirection": "°",
                "weathercode": "wmo code"
            },
            "current_weather": {
                "time": "2026-10-23T19:45",
                "temperature": 13.5,
                "windspeed": 21.4,
                "winddirection": 240,
                "is_day": 0,
                "weathercode": 63
            }
        }
    },
    {
        "title": "Scenario 5: Warm Thunderstorm Afternoon in Miami, USA (-04:00)",
        "user_name": "Elena",
        "location_payload": {
            "city": "Miami",
            "region": "Florida",
            "country": "US",
            "country_name": "United States",
            "latitude": 25.7617,
            "longitude": -80.1918,
            "utc_offset": "-04:00",
            "timezone": "America/New_York"
        },
        "weather_payload": {
            "latitude": 25.7617,
            "longitude": -80.1918,
            "timezone": "GMT",
            "utc_offset_seconds": 0,
            "current_weather_units": {
                "time": "iso8601",
                "temperature": "°C",
                "windspeed": "km/h",
                "winddirection": "°",
                "weathercode": "wmo code"
            },
            "current_weather": {
                "time": "2026-10-23T19:00",
                "temperature": 29.8,
                "windspeed": 34.0,
                "winddirection": 160,
                "is_day": 1,
                "weathercode": 95
            }
        }
    }
]

for sc in scenarios:
    def mock_requests_get(url, params=None, timeout=None):
        resp = MagicMock()
        resp.raise_for_status = MagicMock()
        if "ipapi.co" in url:
            resp.json.return_value = sc["location_payload"]
        elif "open-meteo.com" in url:
            resp.json.return_value = sc["weather_payload"]
        else:
            raise ValueError(f"Unexpected URL: {url}")
        return resp

    with patch("components.nodes.requests.get", side_effect=mock_requests_get):
        initial_state = WeatherAgentState(
            name=sc["user_name"],
            location_data=None,
            weather_data=None,
            weather_info=None
        )
        final_state = weather_agent.invoke(initial_state)
        print("=" * 68)
        print(sc["title"])
        print("=" * 68)
        print(final_state["weather_info"])
        print()

Scenario 1: Comfortable Morning in Bengaluru, India (+0530)
Time: 04:00 UTC | 09:30 (UTC+0530)

Good morning, Aarav!

Your current location: Bengaluru, Karnataka, India

Current weather conditions:
• Partly cloudy
• Temperature: 23.4°C (comfortable)
• Wind: 11.2 km/h

Scenario 2: Freezing Snowy Night in Reykjavik, Iceland (+00:00)
Time: 22:15 UTC | 22:15 (UTC+00:00)

Good evening, Katrin!

Your current location: Reykjavik, Capital Region, Iceland

Current weather conditions:
• Moderate snow fall
• Temperature: -2.8°C (cold)
• Wind: 28.5 km/h

Scenario 3: Hot Daytime in Dubai, UAE (+04:00)
Time: 09:30 UTC | 13:30 (UTC+04:00)

Good morning, Zayd!

Your current location: Dubai, Dubai, United Arab Emirates

Current weather conditions:
• Clear sky
• Temperature: 36.2°C (hot)
• Wind: 18.0 km/h

Scenario 4: Cool Rainy Evening in London, UK (+01:00)
Time: 19:45 UTC | 20:45 (UTC+01:00)

Good evening, Oliver!

Your current location: London, England, United Kingdom

Current weather conditions:
• 

## 6. Edge-Case & Error-Handling Verification

Finally, we test:
1. **`main.py` CLI flow** with blank username input (verifying fallback to `"User"` and `__name__ == "__main__"`).
2. **Missing required location field validation** (verifying descriptive error messages when an upstream API returns incomplete JSON).
3. **Network timeout / HTTP failure handling** (verifying graceful exception handling in `main.py`).

In [3]:
import main
import requests

# Test 1: Blank user input defaults to 'User' in main.py
def mock_get_success(url, params=None, timeout=None):
    resp = MagicMock()
    resp.raise_for_status = MagicMock()
    if "ipapi.co" in url:
        resp.json.return_value = scenarios[0]["location_payload"]
    else:
        resp.json.return_value = scenarios[0]["weather_payload"]
    return resp

print("--- Edge Case 1: Empty Name Input Fallback in main.main() ---")
with patch("builtins.input", return_value="   "), patch("components.nodes.requests.get", side_effect=mock_get_success):
    main.main()

# Test 2: Incomplete API payload validation
print("\n--- Edge Case 2: Incomplete Location Payload Validation ---")
def mock_get_incomplete(url, params=None, timeout=None):
    resp = MagicMock()
    resp.raise_for_status = MagicMock()
    resp.json.return_value = {"city": "TestCity"}  # Missing region, country_name, lat/lon, etc.
    return resp

with patch("builtins.input", return_value="Alex"), patch("components.nodes.requests.get", side_effect=mock_get_incomplete):
    main.main()

# Test 3: Network timeout handling
print("\n--- Edge Case 3: Network Timeout Graceful Error Handling ---")
def mock_get_timeout(url, params=None, timeout=None):
    raise requests.exceptions.ConnectTimeout("Connection to api.open-meteo.com timed out")

with patch("builtins.input", return_value="Alex"), patch("components.nodes.requests.get", side_effect=mock_get_timeout):
    main.main()

--- Edge Case 1: Empty Name Input Fallback in main.main() ---

WEATHER INFORMATION
Time: 04:00 UTC | 09:30 (UTC+0530)

Good morning, User!

Your current location: Bengaluru, Karnataka, India

Current weather conditions:
• Partly cloudy
• Temperature: 23.4°C (comfortable)
• Wind: 11.2 km/h

--- Edge Case 2: Incomplete Location Payload Validation ---

Error: Invalid location data received: Missing required field: region
Please check your internet connection and try again.

--- Edge Case 3: Network Timeout Graceful Error Handling ---

Error: Failed to fetch location data: Connection to api.open-meteo.com timed out
Please check your internet connection and try again.

## 7. Live Execution Test (Real-Time IP Geolocation & Open-Meteo API)

Run the cell below in an internet-connected environment to fetch your live IP location from `ipapi.co` and current weather conditions from `api.open-meteo.com` using the compiled `weather_agent`.

In [4]:
from graph import weather_agent
from components.state import WeatherAgentState

state = WeatherAgentState(
    name="Developer",
    location_data=None,
    weather_data=None,
    weather_info=None
)

final_state = weather_agent.invoke(state)
print("=" * 60)
print("WEATHER INFORMATION")
print("=" * 60)
print(final_state["weather_info"])

WEATHER INFORMATION
Time: 13:25 UTC | 18:55 (UTC+0530)

Good evening, Developer!

Your current location: Bengaluru, Karnataka, India

Current weather conditions:
• Mainly clear
• Temperature: 24.8°C (comfortable)
• Wind: 9.4 km/h